# Libraries

In [33]:
import pandas as pd

# Importing Data

In [61]:
df = pd.read_parquet('../data/eda.parquet')

df.head()

,trans_num,cc_num,ts,trans_date,hour,day_of_week,merchant,category,gender,age_at_txn,...,sec_since_last_txn,is_first_txn,cnt_txn_1h,cnt_txn_24h,cnt_txn_7d,sum_amount_24h,ratio_to_30d_avg,cnt_distinct_merchants_24h,ratio_to_30d_avg_buckets,distance_km_bucket
0,f5cec95f35959f3965bd477ca69b4245,180000273008690,2019-01-08 06:56:54,2019-01-08,6.0,2.0,"Reilly, Heaney and Cole",gas_transport,F,36,...,5518.0,0,0,4,37,530.76,1.6771,3,2. 1x - 3x,3. 50 - 100 km
1,b25bb9df69c8822cd8be7e9f26819393,180000273008690,2019-01-13 09:19:37,2019-01-13,9.0,0.0,"Schaefer, Maggio and Daugherty",gas_transport,F,36,...,5852.0,0,0,6,35,608.33,1.7718,6,2. 1x - 3x,2. 10 - 50 km (nearby towns)
2,788194a9e52d1f586c4a83e1f32cfa12,180000273008690,2019-02-03 00:24:44,2019-02-03,0.0,0.0,Cummings LLC,gas_transport,F,37,...,50919.0,0,0,6,28,615.98,1.4312,6,2. 1x - 3x,3. 50 - 100 km
3,d9618ab186d5afabb88cad2ff2e4fd7b,180000273008690,2019-02-17 08:21:48,2019-02-17,8.0,0.0,Conroy-Cruickshank,gas_transport,F,37,...,20885.0,0,0,5,29,653.12,3.0709,5,3. 3x - 5x,3. 50 - 100 km
4,c34441461341d2e05beb01f6eca85991,180000273008690,2019-03-01 09:08:42,2019-03-01,9.0,5.0,"Ferry, Reichel and DuBuque",grocery_net,F,37,...,9425.0,0,0,5,17,772.48,1.5346,5,2. 1x - 3x,3. 50 - 100 km


In [62]:
df = df.astype({
    'category': 'category',
    'gender': 'category',
    'state': 'category',
    'hour': 'int',
    'day_of_week': 'int'
})

In [63]:
fcols = df.select_dtypes(include=["floating"]).columns
icols = df.select_dtypes(include=["integer"]).columns

df[fcols] = df[fcols].apply(pd.to_numeric, downcast="float")
df[icols] = df[icols].apply(pd.to_numeric, downcast="integer")

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2766578 entries, 0 to 2766577
Data columns (total 31 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   trans_num                   object        
 1   cc_num                      int64         
 2   ts                          datetime64[ns]
 3   trans_date                  datetime64[ns]
 4   hour                        int8          
 5   day_of_week                 int8          
 6   merchant                    object        
 7   category                    category      
 8   gender                      category      
 9   age_at_txn                  int8          
 10  age_bucket                  object        
 11  job                         object        
 12  city                        object        
 13  state                       category      
 14  zip                         int32         
 15  city_pop                    int32         
 16  distance_km            

# 1. Column Selection

Currently there are 33 columns. We need to keep 23: 19 features + 3 utility columns + target.

### Remove completely (10 columns)

| Column | Why |
|---|---|
| `trans_date` | duplicates `ts` |
| `cc_num`, `job`, `city`, `zip` | are tied to a specific client. There are only **998** cards, and each has exactly one city/job/zip code — the model will simply remember specific cards instead of learning to recognize fraud patterns. This model won't work on new clients. |
| `merchant` | 693 categories. This feature is potentially useful, but requires a separate approach (frequency encoding, trained only on the train) — we'll save this for later as an experiment. |
| `age_bucket`, `amt_bucket`, `ratio_to_30d_avg_buckets`, `distance_km_bucket` | these are the groupings you created for the graphs in EDA. The model doesn't need them—it already has the original numerical columns, and it will find better boundaries on its own than those you specify manually. |

In [65]:
df.drop(columns=[
    "trans_date",
    "cc_num",
    "job",
    "city",
    "zip",
    "merchant",
    "age_bucket",
    "amt_bucket",
    "ratio_to_30d_avg_buckets",
    "distance_km_bucket"
], inplace=True)

In [66]:
NUM_FEATURES = ['hour', 'day_of_week', 'age_at_txn', 'city_pop', 'distance_km', 'amt', 'sec_since_last_txn', 'is_first_txn', 'cnt_txn_1h', 'cnt_txn_24h', 'cnt_txn_7d', 'sum_amount_24h', 'ratio_to_30d_avg', 'cnt_distinct_merchants_24h']
CAT_FEATURES = ['category', 'gender', 'state']
TARGET = ['is_fraud']
SERVICE = ['trans_num', 'ts', 'split_period']

df = df[NUM_FEATURES + CAT_FEATURES + TARGET + SERVICE]

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2766578 entries, 0 to 2766577
Data columns (total 21 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   hour                        int8          
 1   day_of_week                 int8          
 2   age_at_txn                  int8          
 3   city_pop                    int32         
 4   distance_km                 float32       
 5   amt                         float64       
 6   sec_since_last_txn          float32       
 7   is_first_txn                int8          
 8   cnt_txn_1h                  int8          
 9   cnt_txn_24h                 int8          
 10  cnt_txn_7d                  int8          
 11  sum_amount_24h              float64       
 12  ratio_to_30d_avg            float32       
 13  cnt_distinct_merchants_24h  int8          
 14  category                    category      
 15  gender                      category      
 16  state                  

# 2. Splitting on train / valid / test

In [67]:
train = df[df.split_period == 'train']
valid = df[df.split_period == 'valid']
test = df[df.split_period == 'test']

In [68]:
assert train.ts.max() < valid.ts.min() < valid.ts.max() < test.ts.min()

In [69]:
assert len(train) + len(valid) + len(test) == len(df)

# 3. Saving

In [70]:
train.to_parquet('../data/processed/train.parquet')
valid.to_parquet('../data/processed/valid.parquet')
test.to_parquet('../data/processed/test.parquet')

In [71]:
import json

feature_schema = {
    "num_features": NUM_FEATURES,
    "cat_features": CAT_FEATURES,
    "target": TARGET,
    "service": SERVICE
}

with open("../data/processed/features.json", "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, ensure_ascii=False, indent=4)